# One-off: rebuild silver CHILDREN from rebuilt parents

**Step 3 of the approved silver rebuild plan - EXECUTED LIVE Nov 9 2026.** All 10 children were written and verified on disk (counts match the dry-run plan exactly). The 7 silver parents were rebuilt to one-row-per-id (see `rebuild_silver_parents_latest_per_id`); this notebook rebuilds every child against those clean parents so each child holds one row per natural key instead of the exploded version history.

Children covered:

* **Leads family**: `leads_processed_custom` (bronze re-parse + latest-per-id dedupe; the nightly task appends every version, ~6x), `leads_processed_contacts` (explode from parent), `leads_processed_contacts_emails` (explode from rebuilt contacts).
* **Activities family**: `lead_activites_email_body` (read from the silver parent instead of the nightly bronze re-parse), `lead_activites_processed_attendees` / `_envelope` / `_to` (explode from parent; 2_2_03/04/05 full-load transforms).
* **Custom activities family** (NEW tables; none exist yet, the 2_4 nightly only writes the parent): `_editable_with_roles`, `_fields`, `_fields_editable_with_roles` from the rebuilt 30-row parent.

Why children needed the parent rebuild first: children explode out of parent rows; rebuilding them against version-log parents would have re-propagated the duplication. Now every explosion is one row per (parent id, element).

Safety:

* `RUN_CHILD_REBUILD = False` by default; every cell is a dry run (build + count, NO writes). Flip to True only after the numbers are reviewed and the live run is explicitly approved.
* Writes are atomic full overwrites (`write_silver` full-load mode: overwrite + overwriteSchema); one cell per child, so a failure stops only that child.
* Run **between** nightlies (outside the ~13:00-13:20 UTC window). Gold layers rebuild after this.
* After the rebuild, child CDC watermarks realign to the parents' bronze stamps (Nov 9 13:01-13:02 loads); the nightly resumes with no-new-data.

In [0]:
%run ../utils


In [0]:
# ============================================================================
# CHILD REBUILD CONFIG - edit flags here, not in the engines
# ============================================================================

# Master switch. False = dry-run (build + count every child, NO writes).
# Flip to True ONLY after the dry-run numbers are reviewed and the live run
# is explicitly approved - same protocol as the parent rebuild notebook.
RUN_CHILD_REBUILD = False

print(f"RUN_CHILD_REBUILD = {RUN_CHILD_REBUILD} "
      f"({'LIVE OVERWRITE' if RUN_CHILD_REBUILD else 'DRY RUN - no writes'})")


In [0]:
# ============================================================================
# CHILD REBUILD ENGINE (1/2) - parent-dedupe parity + explosion parity
# ============================================================================
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, explode_outer, lit
import time

CHILD_RESULTS = []


def latest_per_pk(df, pk):
    """Latest version per PK - the SAME ordering as the parent rebuild
    (bronze_insert_date DESC NULLS LAST, bronze_updated_at DESC NULLS LAST,
    payload_hash DESC), so bronze-parsed children explode from exactly the
    rows the rebuilt parents hold."""
    w = Window.partitionBy(pk).orderBy(
        col("bronze_insert_date").desc_nulls_last(),
        col("bronze_updated_at").desc_nulls_last(),
        col("payload_hash").desc())
    return df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")


In [0]:
# ============================================================================
# CHILD REBUILD ENGINE (2/2) - explode_to_child's transform, returned not
# written; finisher reports counts and (on run) overwrites atomically.
# ============================================================================


def build_child(parent_df, parent_name, array_col, pk, carry_cols=None):
    """explode_to_child's core transform, RETURNED instead of written.
    Same carry / alias / sanitize / flatten logic as utils, so rebuilt
    children match the nightly-written schemas exactly. No recursion:
    call it again on the result for grandchildren (pass carry_cols)."""
    carry_cols = carry_cols or ["bronze_insert_date", "bronze_updated_at", pk]
    df_child = parent_df.withColumn("_element", explode_outer(col(array_col)))
    element_type = df_child.schema["_element"].dataType
    if (isinstance(element_type, StructType)):
        element_field_names = [f.name for f in element_type.fields]
        elem_names = _dedup_names([_sanitize_col_name(f.name) for f in element_type.fields])
        carry = []
        for c in dict.fromkeys(carry_cols):
            if c in parent_df.columns and c != array_col:
                if c in element_field_names or c == pk:
                    carry.append(col(c).alias(f"{parent_name}_{c}"))
                else:
                    carry.append(col(c))
        element_fields = [col(f"_element.`{f.name}`").alias(s)
                          for f, s in zip(element_type.fields, elem_names)]
        df_child = df_child.select(*carry, *element_fields)
    else:
        carry = []
        for c in dict.fromkeys(carry_cols):
            if c in parent_df.columns and c != array_col:
                if c == pk:
                    carry.append(col(c).alias(f"{parent_name}_{c}"))
                else:
                    carry.append(col(c))
        df_child = df_child.select(*carry, col("_element").alias(array_col))
    return flatten_structs(df_child, skip_custom=True)


In [0]:
# ============================================================================
# CHILD REBUILD ENGINE (3/3) - the finisher
# ============================================================================


def finish_child(df, table_name, key_cols=()):
    """Report planned rows + distinct keys; when RUN_CHILD_REBUILD, atomic
    full overwrite via write_silver full-load mode (_CDC_WATERMARK=None)."""
    global _CDC_WATERMARK
    _CDC_WATERMARK = None
    run = RUN_CHILD_REBUILD
    try:
        before = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{table_name}").count()
    except Exception:
        before = None
    n = df.count()
    keys = {k: df.select(k).distinct().count() for k in key_cols if k in df.columns}
    b = f"{before:,} -> " if before is not None else "(new) "
    extra = f" | distinct {keys}" if keys else ""
    print(f"  [count] {table_name}: {b}{n:,} planned rows{extra}")
    if not run:
        print(f"  [DRY] no write. (RUN_CHILD_REBUILD=True overwrites {table_name})")
        r = {"table": table_name, "before": before, "planned": n, "keys": keys,
             "written": False}
    else:
        write_silver(df, table_name)
        after = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{table_name}").count()
        print(f"  [OK] {table_name}: {b}{after:,} rows written")
        r = {"table": table_name, "before": before, "planned": n, "after": after,
             "keys": keys, "written": True}
    CHILD_RESULTS.append(r)
    for temp in _TEMP_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {temp}")
    _TEMP_TABLES.clear()
    return r


In [0]:
# ============================================================================
# LEADS CUSTOM CHILD: leads_processed_custom
# The silver parent DROPS the custom struct (extract_custom turns it into the
# custom_fields MAP), so this child must come from BRONZE full history - but
# deduped to latest-per-id FIRST (the nightly 2_1_02 task appends every
# version, which is the ~6x duplication being fixed here).
# Result: one row per lead, non-cf_ custom fields as columns.
# ============================================================================
print("\n" + "=" * 80)
print("leads_raw -> leads_processed_custom (one row per lead id)")
print("=" * 80)

t0 = time.time()
try:
    before_custom = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.leads_processed_custom").count()
except Exception:
    before_custom = None

df_flat = parse_bronze_table('leads_raw', watermark=None, silver_name=None)
df_flat = flatten_structs(df_flat, skip_custom=True)
df_latest = latest_per_pk(df_flat, 'id')
n_parent = df_latest.count()
print(f"  deduped parent domain: {n_parent:,} leads")


In [0]:
# write path for leads_processed_custom (extract_custom writes the child)
carry = list(dict.fromkeys(["bronze_insert_date", "bronze_updated_at", 'id']))
_CDC_WATERMARK = None  # full-load write inside extract_custom
b = f"{before_custom:,} -> " if before_custom is not None else "(new) "
if RUN_CHILD_REBUILD:
    extract_custom(df_latest, 'leads_processed', carry, "1_1",
                   parent_pk='id', dry_run=False)
    after = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.leads_processed_custom").count()
    print(f"  [OK] leads_processed_custom: {b}{after:,} rows (one per lead)")
    CHILD_RESULTS.append({"table": "leads_processed_custom",
                          "before": before_custom, "planned": n_parent,
                          "after": after, "written": True})
else:
    extract_custom(df_latest, 'leads_processed', carry, "1_1",
                   parent_pk='id', dry_run=True)
    print(f"  [DRY] leads_processed_custom: {b}{n_parent:,} planned rows "
          f"(1:1 with the deduped parent; schema printed above)")
    CHILD_RESULTS.append({"table": "leads_processed_custom",
                          "before": before_custom, "planned": n_parent,
                          "written": False})

for temp in _TEMP_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {temp}")
_TEMP_TABLES.clear()
print(f"  took {time.time() - t0:.1f}s")


In [0]:
# ============================================================================
# LEADS CONTACTS CHILD: leads_processed_contacts
# Explode the contacts array from the REBUILT silver parent (one row per lead
# now) -> one row per (lead, contact). Mirrors 2_1_03_01's explode_to_child
# call (recurse=False; phones/urls/integration_links grandchildren stay dead).
# ============================================================================
print("\n" + "=" * 80)
print("leads_processed -> leads_processed_contacts")
print("=" * 80)

df_parent = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.leads_processed")
CONTACTS_DF = build_child(df_parent, 'leads_processed', 'contacts', 'id')
finish_child(CONTACTS_DF, 'leads_processed_contacts',
             key_cols=['leads_processed_id', 'id'])


In [0]:
# ============================================================================
# LEADS EMAILS GRANDCHILD: leads_processed_contacts_emails
# 2_1_03_02's transform, but sourced from the in-memory rebuilt contacts
# (so dry-run counts are not polluted by the stale version-exploded table).
# ============================================================================
print("\n" + "=" * 80)
print("leads_processed_contacts -> leads_processed_contacts_emails")
print("=" * 80)

df_emails = (
    CONTACTS_DF
    .select(
        col("leads_processed_id"),
        col("id").alias("leads_processed_contacts_id"),
        col("bronze_insert_date"),
        col("bronze_updated_at"),
        explode_outer(col("emails")).alias("email_data"),
    )
    .select(
        "leads_processed_id", "leads_processed_contacts_id",
        "bronze_insert_date", "bronze_updated_at",
        "email_data.*",
    )
)
finish_child(df_emails, 'leads_processed_contacts_emails',
             key_cols=['leads_processed_id', 'leads_processed_contacts_id'])


In [0]:
# ============================================================================
# ACTIVITIES EMAIL BODY CHILD: lead_activites_email_body
# The nightly task re-parses BRONZE per version (no id dedupe) - the root
# cause of this child's duplication. Rebuilt instead from the one-row-per-id
# silver parent: filter type=Email, take body_text_quoted[0].text (2_2_02's
# exact transform), one row per email activity.
# ============================================================================
print("\n" + "=" * 80)
print("lead_activites_processed -> lead_activites_email_body")
print("=" * 80)

df_parent = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.lead_activites_processed")
if "body_text_quoted" not in df_parent.columns:
    print("  [ABORT] body_text_quoted missing from the parent - investigate first; no write.")
else:
    df_email_body = (
        df_parent.filter(col("type") == "Email")
        .select(
            col("bronze_insert_date"), col("bronze_updated_at"),
            col("id").alias("activity_id"), col("lead_id"), col("activity_at"),
            col("status"), col("date_updated"),
            col("body_text_quoted")[0]["text"].alias("body_text"),
        )
        .filter(col("body_text").isNotNull())
    )
    finish_child(df_email_body, 'lead_activites_email_body',
                 key_cols=['activity_id'])


In [0]:
# ============================================================================
# ACTIVITIES ATTENDEES CHILD: lead_activites_processed_attendees
# 2_2_03's full-load transform on the rebuilt parent. Its window dedupe is a
# no-op now that the parent holds one row per id - kept for parity with the
# nightly task.
# ============================================================================
print("\n" + "=" * 80)
print("lead_activites_processed -> lead_activites_processed_attendees")
print("=" * 80)

_version = (Window.partitionBy('id')
            .orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc()))
df_in = (spark.table(f"{CATALOG}.{SILVER_SCHEMA}.lead_activites_processed")
         .select('id', 'lead_id', 'activity_at', 'type',
                 'bronze_insert_date', 'bronze_updated_at', 'attendees')
         .withColumn('_v', row_number().over(_version))
         .filter(col('_v') == 1).drop('_v'))

df_attendees = (
    df_in.select(
        col('id').alias('activity_id'), 'lead_id', 'activity_at', 'type',
        'bronze_insert_date', 'bronze_updated_at',
        explode_outer('attendees').alias('att'),
    )
    .select(
        'activity_id', 'lead_id', 'activity_at', 'type',
        'bronze_insert_date', 'bronze_updated_at',
        col('att.contact_id').alias('contact_id'),
        col('att.email').alias('email'),
        col('att.is_organizer').alias('is_organizer'),
        col('att.name').alias('name'),
        col('att.status').alias('status'),
        col('att.user_id').alias('user_id'),
    )
    .dropna(subset=['contact_id', 'email', 'name', 'user_id'], how='all')
)
finish_child(df_attendees, 'lead_activites_processed_attendees',
             key_cols=['activity_id'])


In [0]:
# ============================================================================
# ACTIVITIES ENVELOPE CHILD: lead_activites_processed_envelope
# 2_2_04's full-load transform on the rebuilt parent, element-shape tolerant
# (bcc flipped array<struct> -> array<string> on Nov 6; either shape parses).
# ============================================================================
print("\n" + "=" * 80)
print("lead_activites_processed -> lead_activites_processed_envelope")
print("=" * 80)

from pyspark.sql.functions import (coalesce, from_json, regexp_extract,
                                   regexp_replace, size, split, trim, when)
from pyspark.sql.types import ArrayType

_version = (Window.partitionBy('id')
            .orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc()))
df_in = (spark.table(f"{CATALOG}.{SILVER_SCHEMA}.lead_activites_processed")
         .select('id', 'lead_id', 'bronze_insert_date', 'bronze_updated_at', 'envelope')
         .withColumn('_v', row_number().over(_version))
         .filter(col('_v') == 1).drop('_v'))


In [0]:
directions = ['from', 'to', 'cc', 'bcc', 'reply_to', 'sender']
env_type = df_in.schema['envelope'].dataType
pieces = []
for d in directions:
    if d not in env_type.fieldNames():
        print(f"  [WARN] direction '{d}' missing from envelope struct - skipped")
        continue
    dir_type = env_type[d].dataType
    elem_is_struct = (isinstance(dir_type, ArrayType)
                      and isinstance(dir_type.elementType, StructType))
    exploded = df_in.select(
        col('id').alias('activity_id'), 'lead_id',
        'bronze_insert_date', 'bronze_updated_at',
        lit(d).alias('direction'),
        explode_outer(col('envelope.' + d)).alias('e'),
        col('envelope.subject').alias('envelope_subject'),
        col('envelope.message_id').alias('envelope_message_id'),
        col('envelope.is_autoreply').alias('envelope_is_autoreply'),
    )
    if elem_is_struct:
        email_expr, name_expr = col('e.email'), col('e.name')
    else:
        # stringified shapes: JSON string, 'Spark cast {email, name}', bare email
        parsed = from_json(col('e'), 'struct<email:string,name:string>')
        is_json_str = col('e').rlike(r'^\s*\{\s*"')
        email_expr = coalesce(parsed['email'],
                              regexp_extract(col('e'), r'[\w.+-]+@[\w.-]+', 0))
        stripped = regexp_replace(col('e'), r'^\s*\{|\}\s*$', '')
        parts = split(stripped, ',')
        name_tok = when(size(parts) > 1, trim(parts[1])).otherwise(
            lit(None).cast('string'))
        name_expr = coalesce(
            when(parsed['name'] != '', parsed['name']),
            when(~is_json_str & (name_tok != ''), name_tok),
        )
    pieces.append(
        exploded.select(
            'activity_id', 'lead_id', 'bronze_insert_date', 'bronze_updated_at',
            'direction',
            email_expr.alias('email'), name_expr.alias('name'),
            'envelope_subject', 'envelope_message_id', 'envelope_is_autoreply',
        )
    )

df_envelope = pieces[0]
for p in pieces[1:]:
    df_envelope = df_envelope.unionByName(p)
df_envelope = df_envelope.dropna(subset=['email']).filter(trim(col('email')) != '')
finish_child(df_envelope, 'lead_activites_processed_envelope',
             key_cols=['activity_id'])


In [0]:
# ============================================================================
# ACTIVITIES 'to' CHILD: lead_activites_processed_to
# 2_2_05's full-load transform on the rebuilt parent (window dedupe = no-op
# parity with the nightly task).
# ============================================================================
print("\n" + "=" * 80)
print("lead_activites_processed -> lead_activites_processed_to")
print("=" * 80)

_version = (Window.partitionBy('id')
            .orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc()))
df_in = (spark.table(f"{CATALOG}.{SILVER_SCHEMA}.lead_activites_processed")
         .select('id', 'lead_id', 'bronze_insert_date', 'bronze_updated_at', 'to')
         .withColumn('_v', row_number().over(_version))
         .filter(col('_v') == 1).drop('_v'))

df_to = (
    df_in.select(
        col('id').alias('activity_id'), 'lead_id',
        'bronze_insert_date', 'bronze_updated_at',
        explode_outer(col('to')).alias('to_email'),
    )
    .dropna(subset=['to_email'])
)
finish_child(df_to, 'lead_activites_processed_to', key_cols=['activity_id'])


In [0]:
# ============================================================================
# CUSTOM ACTIVITIES CHILDREN (NEW TABLES - none exist yet; the 2_4 nightly
# only writes the parent, children were dry-run only):
#   custom_activites_processed_editable_with_roles        (array<string>)
#   custom_activites_processed_fields                     (array<struct>)
#   custom_activites_processed_fields_editable_with_roles (grandchild)
# NOTE: no nightly task maintains these yet - decide later whether to add
# child tasks to 2_4 or treat them as static (definitions rarely change).
# ============================================================================
print("\n" + "=" * 80)
print("custom_activites_processed -> 3 children (new tables)")
print("=" * 80)

df_ca = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.custom_activites_processed")

df_ca_roles = build_child(df_ca, 'custom_activites_processed',
                          'editable_with_roles', 'id')
finish_child(df_ca_roles, 'custom_activites_processed_editable_with_roles',
             key_cols=['custom_activites_processed_id'])

df_ca_fields = build_child(df_ca, 'custom_activites_processed', 'fields', 'id')
finish_child(df_ca_fields, 'custom_activites_processed_fields',
             key_cols=['custom_activites_processed_id', 'id'])

# grandchild: fields.editable_with_roles - carry the grandparent FK down
df_ca_fields_roles = build_child(
    df_ca_fields, 'custom_activites_processed_fields', 'editable_with_roles', 'id',
    carry_cols=['bronze_insert_date', 'bronze_updated_at',
                'custom_activites_processed_id', 'id'])
finish_child(df_ca_fields_roles,
             'custom_activites_processed_fields_editable_with_roles',
             key_cols=['custom_activites_processed_fields_id'])


In [0]:
# ============================================================================
# SUMMARY - one line per child; review before authorizing the live run.
# ============================================================================
print("\n" + "=" * 80)
print(f"CHILD REBUILD SUMMARY  ({'LIVE' if RUN_CHILD_REBUILD else 'DRY RUN'})")
print("=" * 80)
for r in CHILD_RESULTS:
    status = "[written]" if r.get("written") else "[dry-run]"
    b = f"{r['before']:,} -> " if r.get("before") is not None else "(new) "
    a = f"{r['after']:,}" if r.get("after") is not None else f"{r['planned']:,} (planned)"
    print(f"  {status}  {r['table']}: {b}{a} rows")

print()
print("After the live run: children stay on parent-driven delete+append CDC (their")
print("nightly tasks delete affected activity/lead ids and re-append from the clean")
print("parents - correct for multi-row-per-parent children). SILVER_MERGE_KEYS")
print("entries only suit single-PK tables; revisit per child if ever needed.")
print("Gold layers pick up the clean children automatically on their next runs.")


In [0]:
# TEMP ON-DISK VERIFICATION - independent recount of all 10 children after
# the live run. Deleted after verification.
_tables = [
    'leads_processed_custom',
    'leads_processed_contacts',
    'leads_processed_contacts_emails',
    'lead_activites_email_body',
    'lead_activites_processed_attendees',
    'lead_activites_processed_envelope',
    'lead_activites_processed_to',
    'custom_activites_processed_editable_with_roles',
    'custom_activites_processed_fields',
    'custom_activites_processed_fields_editable_with_roles',
]
for _t in _tables:
    _n = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{_t}").count()
    print(f"{_t}: {_n:,}")